# Embed arXiv CS Papers (GPU)

**Mục tiêu:** Encode title + abstract → 384-dim vectors cho toàn bộ CS papers.

**Yêu cầu:**
- Kaggle: chọn GPU T4 x2 (Settings → Accelerator → GPU T4 x2)
- Colab: Runtime → Change runtime type → T4 GPU

**Input:** Upload file `arxiv_cs_100k_clean.json` (100k) hoặc `arxiv_cs_full.jsonl` (1.16M)

**Output:** File JSONL với embedding vectors, download về local

## 1. Setup & Install

In [ ]:
!pip install -q sentence-transformers torch

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f"VRAM: {vram / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU detected! Go to Runtime > Change runtime type > GPU")

## 2. Upload Data

### Option A: Kaggle
- Add dataset: chọn "Upload" → upload file JSON/JSONL
- Hoặc dùng Kaggle API để add arXiv dataset trực tiếp

### Option B: Colab
- Chạy cell bên dưới để upload từ local
- Hoặc mount Google Drive nếu file đã có trên Drive

In [ ]:
# ====== CONFIG (chỉ sửa vài dòng) ======
# USE_DRIVE=True nếu bạn đã để file trên Google Drive.
USE_DRIVE = False
DATA_DIR = "/content/drive/MyDrive/BigData" if USE_DRIVE else "/content"

# Chọn input/output cho full 1.16M
INPUT_NAME = "arxiv_cs_full.jsonl"
OUTPUT_NAME = "arxiv_cs_full_with_vectors.jsonl"

import os

if USE_DRIVE:
    # Colab mount drive (Kaggle sẽ không chạy được đoạn này, nên bọc try)
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print(f"WARNING: drive.mount failed: {e}")

INPUT_FILE = os.path.join(DATA_DIR, INPUT_NAME)
OUTPUT_FILE = os.path.join(DATA_DIR, OUTPUT_NAME)

print(f"Input:  {INPUT_FILE} ({'exists' if os.path.exists(INPUT_FILE) else 'NOT FOUND'})")
if os.path.exists(INPUT_FILE):
    size_mb = os.path.getsize(INPUT_FILE) / (1024*1024)
    print(f"Size:   {size_mb:.1f} MB")

# Progress file để resume nhanh (không cần đếm toàn bộ output mỗi lần)
PROGRESS_FILE = OUTPUT_FILE + ".progress.json"
print(f"Progress: {PROGRESS_FILE}")

## 3. Load Model

In [ ]:
from sentence_transformers import SentenceTransformer
import time

MODEL_NAME = 'all-MiniLM-L6-v2'

model = SentenceTransformer(MODEL_NAME)
print(f"Model: {MODEL_NAME}")
print(f"Device: {model.device}")
print(f"Embedding dim: {model.get_sentence_embedding_dimension()}")

## 4. Quick Sanity Check

In [ ]:
from sentence_transformers.util import cos_sim

test_pairs = [
    ('deep learning for image classification', 'using neural networks to categorize pictures'),
    ('reinforcement learning in robotics', 'training robots with reward-based algorithms'),
    ('natural language processing', 'quantum computing algorithms'),
]

for a, b in test_pairs:
    emb = model.encode([a, b])
    sim = cos_sim(emb[0], emb[1]).item()
    print(f"  {sim:.4f}  |  \"{a}\"  vs  \"{b}\"")

## 5. Run Embedding (Main)

- **GPU T4**: ~512 docs/batch, ~500-800 docs/sec → 100k in ~2-3 min, 1.16M in ~25-35 min
- Checkpoint mỗi 50k records (phòng crash)
- Hỗ trợ resume nếu bị ngắt

In [ ]:
import json
import os
from tqdm.auto import tqdm

BATCH_SIZE = 512  # GPU optimal
CHECKPOINT_EVERY = 50_000

# Progress file để resume nhanh (tránh đếm toàn bộ output JSONL mỗi lần)
PROGRESS_FILE = OUTPUT_FILE + ".progress.json"

def read_records(path):
    """Read JSON array or JSONL file."""
    if path.endswith('.jsonl'):
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    yield json.loads(line)
    else:
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        for record in data:
            yield record

def count_lines(path):
    if path.endswith('.jsonl'):
        with open(path, 'r', encoding='utf-8') as f:
            return sum(1 for _ in f)
    else:
        with open(path, 'r', encoding='utf-8') as f:
            return len(json.load(f))

def count_existing(path):
    if not os.path.exists(path):
        return 0
    with open(path, 'r', encoding='utf-8') as f:
        return sum(1 for _ in f)

# Count input
print("Counting records...")
total = count_lines(INPUT_FILE)

already_done = 0
if os.path.exists(PROGRESS_FILE):
    try:
        with open(PROGRESS_FILE, 'r', encoding='utf-8') as pf:
            state = json.load(pf)
        already_done = int(state.get('already_done', 0))
        print(f"Loaded progress from {PROGRESS_FILE}: already_done={already_done:,}")
    except Exception as e:
        print(f"WARNING: Failed to read progress file: {e}")
        already_done = count_existing(OUTPUT_FILE)
else:
    already_done = count_existing(OUTPUT_FILE)
    if already_done > 0:
        print(f"No progress file found; counted output lines: {already_done:,}")

if already_done > total:
    print(f"WARNING: already_done ({already_done:,}) > total ({total:,}); setting already_done=total")
    already_done = total

print(f"Total: {total:,}  |  Already done: {already_done:,}  |  To process: {total - already_done:,}")

In [ ]:
# === MAIN EMBEDDING LOOP ===
import os

texts_batch = []
records_batch = []
processed_total = already_done
skipped = 0
start_time = time.time()

out_file = open(OUTPUT_FILE, 'a', encoding='utf-8')
pbar = tqdm(total=total, initial=already_done, desc='Embedding', unit='doc')

PROGRESS_FILE = OUTPUT_FILE + ".progress.json"

try:
    for record in read_records(INPUT_FILE):
        skipped += 1
        if skipped <= already_done:
            continue

        title = record.get('title', '')
        abstract = record.get('abstract', '')
        text = f"{title} [SEP] {abstract}"

        texts_batch.append(text)
        records_batch.append(record)

        if len(texts_batch) >= BATCH_SIZE:
            embeddings = model.encode(
                texts_batch,
                show_progress_bar=False,
                normalize_embeddings=True,
                batch_size=BATCH_SIZE,
            )
            for rec, emb in zip(records_batch, embeddings):
                rec['embedding'] = emb.tolist()
                out_file.write(json.dumps(rec, ensure_ascii=False) + '\n')

            processed_total += len(texts_batch)
            pbar.update(len(texts_batch))
            texts_batch.clear()
            records_batch.clear()

            if processed_total % CHECKPOINT_EVERY == 0:
                out_file.flush()
                with open(PROGRESS_FILE, 'w', encoding='utf-8') as pf:
                    json.dump({'already_done': processed_total}, pf)
                elapsed = time.time() - start_time
                speed = processed_total / elapsed if elapsed > 0 else 0
                print(
                    f"  Checkpoint: {processed_total:,} done, {speed:.0f} docs/sec, {elapsed:.0f}s elapsed"
                )

    # Flush remaining
    if texts_batch:
        embeddings = model.encode(
            texts_batch,
            show_progress_bar=False,
            normalize_embeddings=True,
            batch_size=BATCH_SIZE,
        )
        for rec, emb in zip(records_batch, embeddings):
            rec['embedding'] = emb.tolist()
            out_file.write(json.dumps(rec, ensure_ascii=False) + '\n')

        processed_total += len(texts_batch)
        pbar.update(len(texts_batch))

    pbar.close()

finally:
    out_file.flush()
    out_file.close()

    # Always save progress at the end (so resume is fast)
    try:
        with open(PROGRESS_FILE, 'w', encoding='utf-8') as pf:
            json.dump({'already_done': processed_total}, pf)
    except Exception:
        pass

elapsed = time.time() - start_time
speed = processed_total / elapsed if elapsed > 0 else 0
print(f"\nDone! {processed_total:,} records in {elapsed:.1f}s ({speed:.0f} docs/sec)")
print(f"Output: {OUTPUT_FILE}")
print(f"Size: {os.path.getsize(OUTPUT_FILE) / (1024**2):.1f} MB")

## 6. Verify Output

In [ ]:
# Verify
verify_count = 0
sample = None
with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        verify_count += 1
        if verify_count == 1:
            sample = json.loads(line)

print(f"Total records in output: {verify_count:,}")
print(f"Output file size: {os.path.getsize(OUTPUT_FILE) / (1024**2):.1f} MB")
print(f"\nSample record keys: {list(sample.keys())}")
print(f"Embedding dim: {len(sample['embedding'])}")
print(f"First 5 values: {sample['embedding'][:5]}")
print(f"\nTitle: {sample['title'][:80]}...")

## 7. Download Result

### Option A: Colab → Download trực tiếp

In [ ]:
# Colab: download trực tiếp
# from google.colab import files
# files.download(OUTPUT_FILE)

# Colab: copy vào Google Drive (nhanh hơn cho file lớn)
# import shutil
# shutil.copy(OUTPUT_FILE, f'/content/drive/MyDrive/BigData/{OUTPUT_FILE}')
# print('Copied to Google Drive!')

### Option B: Kaggle → Download output

In [ ]:
# Kaggle: copy vào /kaggle/working/ (sẽ xuất hiện trong Output tab)
import shutil
output_dir = '/kaggle/working/'
if os.path.exists(output_dir):
    shutil.copy(OUTPUT_FILE, os.path.join(output_dir, OUTPUT_FILE))
    print(f'Copied to {output_dir} → go to Output tab to download')
else:
    print('Not on Kaggle. Use Colab download method instead.')

---
## Ghi chú quan trọng

| Platform | GPU | VRAM | Session limit | Lưu ý |
|---|---|---|---|---|
| **Kaggle** | T4 (x2) | 16GB | 30h/tuần | Stable, không tự ngắt |
| **Colab Free** | T4 | 15GB | ~12h liên tục | Có thể bị ngắt giữa chừng |
| **Colab Pro** | T4/A100 | 15-40GB | Dài hơn | Trả phí |

**Ước tính thời gian (GPU T4):**
- 100k papers: **~2-3 phút**
- 1.16M papers: **~25-35 phút**

**Nếu bị ngắt giữa chừng:**
- Script có checkpoint/resume
- Chỉ cần chạy lại cell 5 + 6, nó sẽ skip records đã xong

**Sau khi download về local:**
1. Đặt file vào `data/` folder
2. Chạy tiếp `index_data_v2.py` để đưa vào Elasticsearch